In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

In [2]:
# ===== CONFIG =====
N = 20000
SEED = 42

V_MAX = 0.5
W_MAX = 1.0

D_STOP = 0.7
D_SLOW = 1.5
D_FAST = 2.5

K_YAW = 1.2
YAW_MAX = 0.7

rng = np.random.default_rng(SEED)

def clamp(x, a, b):
    return np.minimum(np.maximum(x, a), b)

def compute_forward(df):
    if df <= D_STOP:
        return 0.0
    if df >= D_FAST:
        v = V_MAX
    elif df <= D_SLOW:
        t = (df - D_STOP) / (D_SLOW - D_STOP)
        v = t * 0.08
    else:
        t = (df - D_SLOW) / (D_FAST - D_SLOW)
        v = 0.08 + t * (V_MAX - 0.08)
    return float(clamp(v, 0.0, V_MAX))

def compute_yaw(dl, dr, df):
    diff = dl - dr
    yaw = -K_YAW * diff
    if df < D_SLOW:
        t = clamp((D_SLOW - df) / (D_SLOW - D_STOP), 0.0, 1.0)
        yaw *= (1.0 + 0.6 * t)
    return float(clamp(yaw, -YAW_MAX, YAW_MAX))

mix = rng.uniform(0, 1, size=N)
df = np.where(
    mix < 0.75,
    rng.normal(loc=2.6, scale=0.6, size=N),
    rng.normal(loc=0.9, scale=0.25, size=N)
)

dl = df + rng.normal(loc=0.2, scale=0.5, size=N)
dr = df + rng.normal(loc=0.2, scale=0.5, size=N)

side_event = rng.uniform(0, 1, size=N)
dl = np.where(side_event < 0.08, rng.normal(0.6, 0.2, size=N), dl)
dr = np.where((side_event >= 0.08) & (side_event < 0.16), rng.normal(0.6, 0.2, size=N), dr)

df = clamp(df, 0.2, 6.0)
dl = clamp(dl, 0.2, 6.0)
dr = clamp(dr, 0.2, 6.0)

df_n = clamp(df + rng.normal(0, 0.03, size=N), 0.2, 6.0)
dl_n = clamp(dl + rng.normal(0, 0.03, size=N), 0.2, 6.0)
dr_n = clamp(dr + rng.normal(0, 0.03, size=N), 0.2, 6.0)

v = np.zeros(N, dtype=np.float32)
w = np.zeros(N, dtype=np.float32)

for i in range(N):
    v[i] = compute_forward(df_n[i])
    w[i] = compute_yaw(dl_n[i], dr_n[i], df_n[i])

v = clamp(v + rng.normal(0, 0.01, size=N), 0.0, V_MAX)
w = clamp(w + rng.normal(0, 0.03, size=N), -W_MAX, W_MAX)

data = pd.DataFrame({
    "df": df_n.astype(np.float32),
    "dl": dl_n.astype(np.float32),
    "dr": dr_n.astype(np.float32),
    "v":  v.astype(np.float32),
    "w":  w.astype(np.float32),
})

data.to_csv("rov_data.csv", index=False)
print("✅ Saved rov_data.csv:", data.shape)
data.head()

✅ Saved rov_data.csv: (20000, 5)


,df,dl,dr,v,w
0,0.556730,0.565971,0.658696,0.000000,0.214148
1,2.354272,2.513188,2.563812,0.446848,0.088190
2,1.064456,2.098536,0.532684,0.040062,-0.660484
3,3.856841,4.532229,4.204085,0.500000,-0.410438
4,2.750585,2.756665,3.491381,0.500000,0.713912


In [3]:
IN_COLS  = ["df", "dl", "dr"]
OUT_COLS = ["v", "w"]

BATCH = 256
EPOCHS = 40
LR = 1e-3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

class ROVDataset(Dataset):
    def __init__(self, df: pd.DataFrame, v_max: float, w_max: float):
        X = df[IN_COLS].values.astype(np.float32)
        y = df[OUT_COLS].values.astype(np.float32)

        # Normalize outputs to [-1, 1]
        y[:, 0] = (y[:, 0] / v_max) * 2.0 - 1.0   # v -> [-1,1]
        y[:, 1] = (y[:, 1] / w_max)               # w -> [-1,1]
        y[:, 1] = np.clip(y[:, 1], -1.0, 1.0)

        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class PolicyNet(nn.Module):
    def __init__(self, in_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return torch.tanh(self.net(x))

Device: cpu


In [4]:
df = pd.read_csv("rov_data.csv").dropna()
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

n = len(df)
n_train = int(0.9 * n)
df_train = df.iloc[:n_train]
df_val   = df.iloc[n_train:]

train_ds = ROVDataset(df_train, V_MAX, W_MAX)
val_ds   = ROVDataset(df_val, V_MAX, W_MAX)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False)

model = PolicyNet(in_dim=len(IN_COLS)).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.SmoothL1Loss()

best_val = 1e9

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        pred = model(X)
        loss = loss_fn(pred, y)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_loss += loss.item() * X.size(0)
    tr_loss /= len(train_ds)

    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            va_loss += loss_fn(pred, y).item() * X.size(0)
    va_loss /= len(val_ds)

    print(f"Epoch {epoch:02d} | train {tr_loss:.5f} | val {va_loss:.5f}")

    if va_loss < best_val:
        best_val = va_loss
        example = torch.zeros(1, len(IN_COLS)).to(DEVICE)
        traced = torch.jit.trace(model, example)
        traced.save("policy_actor.pt")
        print("  ✅ Saved policy_actor.pt (best so far)")

print("✅ Done. Best val loss:", best_val)

Epoch 01 | train 0.10558 | val 0.02713
  ✅ Saved policy_actor.pt (best so far)
Epoch 02 | train 0.01144 | val 0.00596
  ✅ Saved policy_actor.pt (best so far)
Epoch 03 | train 0.00504 | val 0.00441
  ✅ Saved policy_actor.pt (best so far)
Epoch 04 | train 0.00397 | val 0.00359
  ✅ Saved policy_actor.pt (best so far)
Epoch 05 | train 0.00319 | val 0.00291
  ✅ Saved policy_actor.pt (best so far)
Epoch 06 | train 0.00255 | val 0.00228
  ✅ Saved policy_actor.pt (best so far)
Epoch 07 | train 0.00192 | val 0.00171
  ✅ Saved policy_actor.pt (best so far)
Epoch 08 | train 0.00157 | val 0.00148
  ✅ Saved policy_actor.pt (best so far)
Epoch 09 | train 0.00142 | val 0.00141
  ✅ Saved policy_actor.pt (best so far)
Epoch 10 | train 0.00134 | val 0.00131
  ✅ Saved policy_actor.pt (best so far)
Epoch 11 | train 0.00127 | val 0.00128
  ✅ Saved policy_actor.pt (best so far)
Epoch 12 | train 0.00122 | val 0.00119
  ✅ Saved policy_actor.pt (best so far)
Epoch 13 | train 0.00117 | val 0.00128
Epoch 14 | tr

In [5]:
policy = torch.jit.load("policy_actor.pt")
policy.eval()

def decode_action(y, v_max=V_MAX, w_max=W_MAX):
    v_norm, w_norm = float(y[0]), float(y[1])
    v = (v_norm + 1.0) * 0.5 * v_max    # [0..V_MAX]
    w = w_norm * w_max                  # [-W_MAX..W_MAX]
    return v, w

tests = [
    np.array([3.0, 3.0, 3.0], dtype=np.float32),  # open space
    np.array([0.8, 2.0, 2.0], dtype=np.float32),  # obstacle front
    np.array([2.0, 0.6, 2.5], dtype=np.float32),  # obstacle left
    np.array([2.0, 2.5, 0.6], dtype=np.float32),  # obstacle right
]

for obs in tests:
    x = torch.from_numpy(obs).unsqueeze(0)
    with torch.no_grad():
        y = policy(x).squeeze(0).numpy()
    v, w = decode_action(y)
    print(f"obs={obs} -> raw={y} -> v={v:.3f}, w={w:.3f}")

obs=[3. 3. 3.] -> raw=[ 0.99640995 -0.0025066 ] -> v=0.499, w=-0.003
obs=[0.8 2.  2. ] -> raw=[-0.9539622   0.00490045] -> v=0.012, w=0.005
obs=[2.  0.6 2.5] -> raw=[0.14113195 0.70625716] -> v=0.285, w=0.706
obs=[2.  2.5 0.6] -> raw=[ 0.12856366 -0.7006361 ] -> v=0.282, w=-0.701


In [6]:
from google.colab import files
files.download("rov_data.csv")
files.download("policy_actor.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>